In [2]:
# ============================================================
# CELL 1: Imports and MLflow Setup
# ============================================================
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import joblib
import warnings
import os

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score
)

warnings.filterwarnings('ignore')
RANDOM_SEED = 42

# Use SQLite backend — recommended for MLflow 2.x+
# This creates a single mlflow.db file in your project root
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("customer_churn_prediction")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment set:      customer_churn_prediction")
print("Backend:             SQLite (mlflow.db)")

2026/06/10 15:49:52 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/10 15:49:53 INFO mlflow.store.db.utils: Updating database tables
2026/06/10 15:49:58 INFO mlflow.tracking.fluent: Experiment with name 'customer_churn_prediction' does not exist. Creating a new experiment.


MLflow tracking URI: sqlite:///../mlflow.db
Experiment set:      customer_churn_prediction
Backend:             SQLite (mlflow.db)


In [3]:
# ============================================================
# CELL 2: Prepare Data (same as churn prediction notebook)
# ============================================================
customers = pd.read_csv(
    '../data/processed/customer_features.csv',
    dtype={'Customer ID': str}
)

le_country = LabelEncoder()
customers['Country_encoded'] = le_country.fit_transform(customers['Country'])

FEATURE_COLS = [
    'Frequency', 'Monetary',
    'AvgOrderValue', 'AvgBasketSize', 'UniqueProducts',
    'Tenure', 'ReturnRate', 'ReturnCount',
    'PreferredDayOfWeek', 'PreferredMonth',
    'Country_encoded'
]

X = customers[FEATURE_COLS]
y = customers['Churned']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=RANDOM_SEED, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f"Data ready: {X_train_scaled.shape[0]:,} train | {X_test_scaled.shape[0]:,} test")

Data ready: 4,702 train | 1,176 test


In [4]:
# ============================================================
# CELL 3: Log All 5 Models as MLflow Experiments
# ============================================================
# Each model run is logged as a separate "experiment run"
# MLflow records: parameters, metrics, model artifacts
#
# What gets logged:
# - params: model hyperparameters (n_estimators, max_depth etc.)
# - metrics: all evaluation scores
# - artifacts: the trained model file itself
# - tags: descriptive labels for filtering runs

models_config = {
    'LogisticRegression': {
        'model': LogisticRegression(
            random_state=RANDOM_SEED, max_iter=1000, C=1.0
        ),
        'params': {'C': 1.0, 'max_iter': 1000, 'solver': 'lbfgs'}
    },
    'RandomForest': {
        'model': RandomForestClassifier(
            n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1
        ),
        'params': {'n_estimators': 100, 'max_depth': 'None'}
    },
    'XGBoost': {
        'model': XGBClassifier(
            n_estimators=100, random_state=RANDOM_SEED,
            eval_metric='logloss', verbosity=0
        ),
        'params': {'n_estimators': 100, 'learning_rate': 0.3}
    },
    'LightGBM': {
        'model': LGBMClassifier(
            n_estimators=100, random_state=RANDOM_SEED, verbose=-1
        ),
        'params': {'n_estimators': 100, 'learning_rate': 0.1}
    },
    'CatBoost': {
        'model': CatBoostClassifier(
            iterations=100, random_state=RANDOM_SEED,
            verbose=0, thread_count=1
        ),
        'params': {'iterations': 100, 'learning_rate': 0.03}
    }
}

print("Logging experiments to MLflow...")
print("-" * 50)

for model_name, config in models_config.items():
    with mlflow.start_run(run_name=model_name):

        model = config['model']

        # Log parameters
        mlflow.log_params(config['params'])
        mlflow.log_param('model_type', model_name)
        mlflow.log_param('n_features', len(FEATURE_COLS))
        mlflow.log_param('train_size', len(X_train_scaled))
        mlflow.log_param('random_seed', RANDOM_SEED)

        # Cross-validation scores
        cv_scores = cross_validate(
            model, X_train_scaled, y_train,
            cv=cv,
            scoring=['accuracy', 'f1', 'roc_auc', 'precision', 'recall'],
            n_jobs=-1
        )

        # Log CV metrics
        mlflow.log_metric('cv_accuracy',  cv_scores['test_accuracy'].mean())
        mlflow.log_metric('cv_f1',        cv_scores['test_f1'].mean())
        mlflow.log_metric('cv_roc_auc',   cv_scores['test_roc_auc'].mean())
        mlflow.log_metric('cv_precision', cv_scores['test_precision'].mean())
        mlflow.log_metric('cv_recall',    cv_scores['test_recall'].mean())

        # Train on full training set and evaluate on test set
        model.fit(X_train_scaled, y_train)
        y_pred       = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

        # Log test metrics
        mlflow.log_metric('test_accuracy',  accuracy_score(y_test, y_pred))
        mlflow.log_metric('test_f1',        f1_score(y_test, y_pred))
        mlflow.log_metric('test_roc_auc',   roc_auc_score(y_test, y_pred_proba))
        mlflow.log_metric('test_precision', precision_score(y_test, y_pred))
        mlflow.log_metric('test_recall',    recall_score(y_test, y_pred))
        mlflow.log_metric('test_pr_auc',    average_precision_score(y_test, y_pred_proba))

        # Log model artifact
        mlflow.sklearn.log_model(model, artifact_path=model_name)

        # Tag the best model
        if model_name == 'CatBoost':
            mlflow.set_tag('status', 'best_model')
            mlflow.set_tag('production_ready', 'true')
        else:
            mlflow.set_tag('status', 'baseline')

        print(f"✓ {model_name:<22} logged | "
              f"Test ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

print("\nAll experiments logged to MLflow.")

Logging experiments to MLflow...
--------------------------------------------------


2026/06/10 15:51:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 15:51:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ LogisticRegression     logged | Test ROC-AUC: 0.8401


2026/06/10 15:51:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 15:51:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ RandomForest           logged | Test ROC-AUC: 0.8850


...

✓ XGBoost                logged | Test ROC-AUC: 0.8895


2026/06/10 15:52:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 15:52:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ LightGBM               logged | Test ROC-AUC: 0.8958


2026/06/10 15:52:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 15:52:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✓ CatBoost               logged | Test ROC-AUC: 0.9004

All experiments logged to MLflow.


In [5]:
# ============================================================
# CELL 4: Query MLflow Results Programmatically
# ============================================================
import mlflow

mlflow.set_tracking_uri("sqlite:///../mlflow.db")

experiment = mlflow.get_experiment_by_name("customer_churn_prediction")
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_roc_auc DESC"]
)

display_cols = [
    'tags.mlflow.runName',
    'metrics.test_accuracy',
    'metrics.test_precision',
    'metrics.test_recall',
    'metrics.test_f1',
    'metrics.test_roc_auc',
    'metrics.test_pr_auc'
]

results = runs[display_cols].copy()
results.columns = [
    'Model', 'Accuracy', 'Precision',
    'Recall', 'F1', 'ROC-AUC', 'PR-AUC'
]
results = results.round(4)

print("=" * 70)
print("MLFLOW EXPERIMENT RESULTS — RANKED BY ROC-AUC")
print("=" * 70)
print(results.to_string(index=False))
print("=" * 70)

print("""
TO VIEW THE MLFLOW UI:
─────────────────────────────────────────────
1. Open a NEW Git Bash terminal in VS Code
2. Activate venv: source venv/Scripts/activate
3. Navigate to project: cd ~/Desktop/customer_analytics
4. Run: mlflow ui --backend-store-uri sqlite:///mlflow.db
5. Open browser: http://127.0.0.1:5000
─────────────────────────────────────────────
""")

MLFLOW EXPERIMENT RESULTS — RANKED BY ROC-AUC
             Model  Accuracy  Precision  Recall     F1  ROC-AUC  PR-AUC
          CatBoost    0.8010     0.8182  0.7826 0.8000   0.9004  0.9090
          LightGBM    0.7891     0.8007  0.7793 0.7898   0.8958  0.9045
           XGBoost    0.7781     0.7890  0.7692 0.7790   0.8895  0.8999
      RandomForest    0.7840     0.7997  0.7676 0.7833   0.8850  0.8932
LogisticRegression    0.7466     0.7259  0.8060 0.7639   0.8401  0.8580

TO VIEW THE MLFLOW UI:
─────────────────────────────────────────────
1. Open a NEW Git Bash terminal in VS Code
2. Activate venv: source venv/Scripts/activate
3. Navigate to project: cd ~/Desktop/customer_analytics
4. Run: mlflow ui --backend-store-uri sqlite:///mlflow.db
5. Open browser: http://127.0.0.1:5000
─────────────────────────────────────────────



In [6]:
# ============================================================
# CELL 5: Production Model Summary
# ============================================================
# In a real MLOps setup this would register the model in
# MLflow Model Registry with staging/production tags.
# Here we document it clearly for portfolio purposes.

os.makedirs('../models/reports', exist_ok=True)

model_report = {
    'model_name': 'CatBoost Churn Predictor',
    'version': '1.0.0',
    'trained_on': '2009-12-01 to 2011-12-09',
    'features': FEATURE_COLS,
    'n_features': len(FEATURE_COLS),
    'test_roc_auc': 0.9004,
    'test_f1': 0.8000,
    'test_precision': 0.8182,
    'test_recall': 0.7826,
    'churn_threshold_days': 90,
    'training_samples': len(X_train_scaled),
    'test_samples': len(X_test_scaled),
    'random_seed': RANDOM_SEED
}

import json
with open('../models/reports/model_card.json', 'w') as f:
    json.dump(model_report, f, indent=4)

print("=" * 50)
print("PRODUCTION MODEL CARD")
print("=" * 50)
for key, value in model_report.items():
    if key != 'features':
        print(f"  {key:<25} {value}")
print(f"  {'features':<25} {len(FEATURE_COLS)} features")
print("=" * 50)
print("\nSaved: models/reports/model_card.json")
print("\nMLflow tracking complete.")
print("Run 'mlflow ui --backend-store-uri mlruns' to view UI")

PRODUCTION MODEL CARD
  model_name                CatBoost Churn Predictor
  version                   1.0.0
  trained_on                2009-12-01 to 2011-12-09
  n_features                11
  test_roc_auc              0.9004
  test_f1                   0.8
  test_precision            0.8182
  test_recall               0.7826
  churn_threshold_days      90
  training_samples          4702
  test_samples              1176
  random_seed               42
  features                  11 features

Saved: models/reports/model_card.json

MLflow tracking complete.
Run 'mlflow ui --backend-store-uri mlruns' to view UI
